# 🧠 DocMind — Multi-LLM RAG Comparison System

## Project Overview

This notebook implements a **Retrieval-Augmented Generation (RAG)** pipeline that runs the same question through **4 different LLM models** simultaneously, enabling direct comparison of their answers, speed, and token usage.

### Problem Statement
> *Given a set of PDF documents, which LLM model provides the most accurate, fastest, and most detailed answer to user queries?*

### Models Compared (all via Groq API)
| # | Model | Family | Parameters | Context Window | Key Strength |
|---|-------|--------|-----------|----------------|--------------|
| 1 | `llama-3.3-70b-versatile` | Meta LLaMA | 70B | 128K | Best overall reasoning |
| 2 | `llama-3.1-8b-instant` | Meta LLaMA | 8B | 8K | Fast & lightweight |
| 3 | `qwen/qwen3-32b` | Mistral AI | 8x7B MoE | 32K | Mixture-of-Experts |
| 4 | `openai/gpt-oss-20b` | Google DeepMind | 9B | 8K | Google efficiency |

### RAG Pipeline
```
PDF Upload → Text Extraction → Chunking → Embeddings (MiniLM) → FAISS Index → Retrieval → LLM × 4 → Comparison
```


In [39]:
# CELL 1: Install dependencies
# Run this once — subsequent cells assume libraries are available
get_ipython().system('pip install pymupdf sentence-transformers faiss-cpu groq numpy --quiet')
print("All dependencies installed")


All dependencies installed


In [40]:
# CELL 2: Imports
import fitz                          # PyMuPDF — PDF text extraction
import numpy as np                   # Numerical operations
import faiss                         # Vector similarity search
import os, time, json
from groq import Groq
from sentence_transformers import SentenceTransformer
print("All libraries imported successfully")


All libraries imported successfully


In [1]:
# CELL 3: Configuration — edit these values before running
GROQ_API_KEY = "<YOUR_GROQ_API_KEY>"       # Paste your Groq API key here

# PDFs to analyse — upload them to Jupyter file browser first
PDF_PATHS = ["Genome annotation.pdf","metagenomics.pdf","insilico drug designing.pdf","microarray data analysis.pdf"]        # Add your PDF filenames here

# Chunking parameters
CHUNK_SIZE    = 500   # Characters per chunk
CHUNK_OVERLAP = 50    # Overlap to preserve context at boundaries
TOP_K         = 6     # Chunks to retrieve per query

# All 4 models to compare — they all run on Groq (free tier available)
MODELS = {
    "llama-3.3-70b-versatile": {
        "label": "LLaMA 3.3 70B",
        "family": "Meta LLaMA",
        "params": "70B",
        "context": "128K tokens",
    },
    "llama-3.1-8b-instant": {
        "label": "LLaMA 3.1 8B",
        "family": "Meta LLaMA",
        "params": "8B",
        "context": "8K tokens",
    },
    "qwen/qwen3-32b": {
        "label": "Qwen3 32B",
        "family": "Mistral AI",
        "params": "8x7B MoE",
        "context": "32K tokens",
    },
    "openai/gpt-oss-20b": {
        "label": "GPT-OSS 20B",
        "family": "Google DeepMind",
        "params": "9B",
        "context": "8K tokens",
    },
}
print(f"Config loaded | {len(MODELS)} models configured")


Config loaded | 4 models configured


In [42]:
# CELL 4: PDF Loading
def load_pdfs(pdf_paths):
    """Extract text from every page of each PDF."""
    pages = []
    for path in pdf_paths:
        if not os.path.exists(path):
            print(f"WARNING: '{path}' not found — skipping")
            continue
        doc   = fitz.open(path)
        fname = os.path.basename(path)
        for page_num, page in enumerate(doc):
            text = page.get_text().strip()
            if text:
                pages.append({
                    "text":   text,
                    "source": fname,
                    "page":   page_num + 1
                })
        print(f"Loaded '{fname}' — {len(doc)} pages")
    print(f"Total pages extracted: {len(pages)}")
    return pages

pages = load_pdfs(PDF_PATHS)


Loaded 'Genome annotation.pdf' — 20 pages
Loaded 'metagenomics.pdf' — 19 pages
Loaded 'insilico drug designing.pdf' — 40 pages
Loaded 'microarray data analysis.pdf' — 29 pages
Total pages extracted: 30


In [43]:
# CELL 5: Text Chunking
def split_into_chunks(pages, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Split each page into overlapping fixed-size character chunks.

    Why overlap?
    Sentences near chunk boundaries might be cut off. The overlap
    window ensures neighbouring chunks share some text, so no
    information is lost at boundary seams.
    """
    chunks = []
    for p in pages:
        text, start = p["text"], 0
        while start < len(text):
            chunks.append({
                "text":   text[start:start + chunk_size],
                "source": p["source"],
                "page":   p["page"]
            })
            start += chunk_size - overlap
    print(f"{len(chunks)} chunks created from {len(pages)} pages")
    print(f"Chunk size: {chunk_size} chars | Overlap: {overlap} chars")
    return chunks

chunks = split_into_chunks(pages)

# Preview
print(f"\nSample chunk from '{chunks[0]['source']}', page {chunks[0]['page']}:")
print(chunks[0]['text'][:300] + "...")


36 chunks created from 30 pages
Chunk size: 500 chars | Overlap: 50 chars

Sample chunk from 'Genome annotation.pdf', page 2:
• The genome contains all the biological information required to build and maintain any given 
living organism.
• The genome contains the organisms molecular history
• Decoding the biological information encoded in these molecules will have enormous impact in 
our understanding of biology
• Three ty...


In [44]:
# CELL 6: Embeddings + FAISS Index
print("Loading sentence-transformer embedding model...")
print("(First run downloads ~90MB — subsequent runs use cache)\n")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded: all-MiniLM-L6-v2")
print("Output dimension: 384 floats per chunk\n")

print("Generating chunk embeddings...")
t0          = time.time()
chunk_texts = [c["text"] for c in chunks]
embeddings  = embed_model.encode(
    chunk_texts,
    show_progress_bar=True,
    batch_size=64
).astype("float32")
elapsed = round(time.time() - t0, 1)
print(f"Embeddings shape: {embeddings.shape} | Time: {elapsed}s")

print("\nBuilding FAISS index (IndexFlatL2 = exact L2 nearest neighbour)...")
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print(f"FAISS index ready — {index.ntotal} vectors indexed")


Loading sentence-transformer embedding model...
(First run downloads ~90MB — subsequent runs use cache)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded: all-MiniLM-L6-v2
Output dimension: 384 floats per chunk

Generating chunk embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (36, 384) | Time: 0.6s

Building FAISS index (IndexFlatL2 = exact L2 nearest neighbour)...
FAISS index ready — 36 vectors indexed


In [45]:
# CELL 7: Retrieval Function
def retrieve(query, top_k=TOP_K, diverse=True):
    """
    Two retrieval strategies:

    diverse=True  (recommended for multi-document queries)
        Searches top 50 globally, then keeps only the BEST
        chunk from each document. Guarantees all documents
        contribute to the answer.

    diverse=False (standard top-K)
        Returns the top_k globally closest chunks — may
        be dominated by one document if it's very relevant.
    """
    qvec = embed_model.encode([query]).astype("float32")

    if diverse:
        D, I = index.search(qvec, min(50, index.ntotal))
        best = {}
        for dist, idx in zip(D[0], I[0]):
            src = chunks[idx]["source"]
            if src not in best or dist < best[src]["dist"]:
                best[src] = {"dist": float(dist), "idx": int(idx)}
        results = sorted(best.values(), key=lambda x: x["dist"])
        return [{**chunks[r["idx"]], "distance": round(r["dist"], 4)} for r in results]
    else:
        D, I = index.search(qvec, top_k)
        return [{**chunks[i], "distance": round(float(d), 4)} for d, i in zip(D[0], I[0])]

# Test retrieval
print("Retrieval test — query: 'main topic'")
test_hits = retrieve("main topic", top_k=3)
for i, h in enumerate(test_hits[:3]):
    print(f"  Chunk {i+1}: {h['source']} p{h['page']} | dist={h['distance']} | {h['text'][:80]}...")


Retrieval test — query: 'main topic'
  Chunk 1: Genome annotation.pdf p3 | dist=1.6372 | The Human genome project 
promised to revolutionise medicine 
and explain every ...
  Chunk 2: insilico drug designing.pdf p31 | dist=1.6638 | Step 3: Selection of QSAR methods
• Computerized methods
1.
Multiple linear regr...
  Chunk 3: microarray data analysis.pdf p2 | dist=1.7784 | Data in such large quantities is difficult - if not impossible - to analyze 
wit...


In [46]:
# CELL 8: Groq API Client + Single-Model Query Function
client = Groq(api_key=GROQ_API_KEY)

def query_model(model_id, prompt):
    """
    Query one Groq-hosted LLM and return structured results:
    - answer  : the model's response text
    - time    : wall-clock response time in seconds
    - tokens  : number of completion tokens generated
    - error   : None if successful, error string if not
    """
    try:
        t_start = time.time()
        resp = client.chat.completions.create(
            model       = model_id,
            messages    = [{"role": "user", "content": prompt}],
            max_tokens  = 1024,
            temperature = 0.2,        # Low temp = deterministic, factual
        )
        elapsed = round(time.time() - t_start, 2)
        return {
            "answer": resp.choices[0].message.content.strip(),
            "time":   elapsed,
            "tokens": resp.usage.completion_tokens if resp.usage else 0,
            "error":  None
        }
    except Exception as e:
        return {"answer": "", "time": 0, "tokens": 0, "error": str(e)}

# Verify connection
test = query_model("llama-3.1-8b-instant", "Reply with only the word: CONNECTED")
print("Connection test:", test["answer"])


Connection test: CONNECTED


In [47]:
# CELL 9: Build RAG Prompt
def build_rag_prompt(query, hits):
    """
    Construct the prompt that will be sent to every LLM.
    All models receive IDENTICAL context — ensuring a fair comparison.

    The prompt instructs the model to:
    1. Answer only from the provided context (grounded RAG)
    2. Cite which source document supports the answer
    """
    context = "\n\n".join(
        f"[Source {i+1}: {h['source']}, Page {h['page']}]\n{h['text']}"
        for i, h in enumerate(hits)
    )
    return f"""You are a precise research assistant. Answer using ONLY the context below.
Cite which source(s) support your answer. Be thorough but concise.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

# Test prompt preview
sample_hits   = retrieve("main topic", top_k=2)
sample_prompt = build_rag_prompt("What is this document about?", sample_hits)
print("Sample prompt (first 500 chars):")
print(sample_prompt[:500] + "...")


Sample prompt (first 500 chars):
You are a precise research assistant. Answer using ONLY the context below.
Cite which source(s) support your answer. Be thorough but concise.

CONTEXT:
[Source 1: Genome annotation.pdf, Page 3]
The Human genome project 
promised to revolutionise medicine 
and explain every base of DNA
Identify variation in the 
genome that is disease 
causing
Determine how individual 
genes play a role in health 
and disease
Large MEDICAL GENETICS focus

[Source 2: insilico drug designing.pdf, Page 31]
Step 3: S...


In [48]:
# CELL 10: Core Comparison Function
def compare_models(query, selected_models=None, diverse=True):
    """
    Run the same RAG query through every selected model.

    All models receive identical context — the only variable is
    the LLM itself. This makes it a controlled experiment.

    Returns:
        dict with per-model results + ranked summary
    """
    if selected_models is None:
        selected_models = list(MODELS.keys())

    # Step 1: Shared retrieval — identical context for all models
    hits   = retrieve(query, diverse=diverse)
    prompt = build_rag_prompt(query, hits)

    print(f"Query: {query}")
    print(f"Context: {len(hits)} chunks retrieved from {len(set(h['source'] for h in hits))} documents")
    print("-" * 60)

    # Step 2: Query each model
    results = {}
    for mid in selected_models:
        label = MODELS[mid]["label"]
        print(f"  Querying {label}...", end=" ", flush=True)
        res = query_model(mid, prompt)
        results[mid] = res
        if res["error"]:
            print(f"ERROR: {res['error'][:60]}")
        else:
            wps = f"{res['tokens']/res['time']:.1f} t/s" if res['time'] > 0 else "—"
            print(f"done | {res['time']}s | {res['tokens']} tokens | {wps}")

    # Step 3: Rank by speed (fastest = rank 1)
    ok     = {k: v for k, v in results.items() if not v["error"]}
    ranked = sorted(ok.keys(), key=lambda k: ok[k]["time"])

    # Step 4: Print comparison table
    print("\n" + "=" * 65)
    print(f"{'Rank':<5} {'Model':<24} {'Time':>7} {'Tokens':>8} {'Speed':>10}")
    print("-" * 65)
    for rank, mid in enumerate(ranked, 1):
        r   = ok[mid]
        wps = f"{r['tokens']/r['time']:.1f} t/s" if r["time"] > 0 else "—"
        print(f"#{rank:<4} {MODELS[mid]['label']:<24} {r['time']:>5.2f}s {r['tokens']:>8} {wps:>10}")
    print("=" * 65)

    return {
        "query":  query,
        "hits":   hits,
        "results": results,
        "ranked": ranked
    }


In [49]:
# CELL 11: Run Multi-Model Comparison
# ─────────────────────────────────────────────────────────────────
# Define your questions here — add/modify as needed
# ─────────────────────────────────────────────────────────────────
QUESTIONS = [
    "What is the main topic of this document?",
    "What are the key findings or conclusions?",
    "What methods or approaches are described?",
    "What are the limitations or challenges mentioned?",
]

all_runs = []

for question in QUESTIONS:
    print("\n" + "=" * 65)
    run = compare_models(question)
    all_runs.append(run)

    # Print each model's answer
    print("\nANSWERS:")
    for mid in run["ranked"]:
        res = run["results"][mid]
        if not res["error"]:
            print(f"\n[ {MODELS[mid]['label']} ]")
            print(res["answer"][:600] + ("..." if len(res["answer"]) > 600 else ""))



Query: What is the main topic of this document?
Context: 3 chunks retrieved from 3 documents
------------------------------------------------------------
done | 1.58s | 190 tokens | 120.3 t/s
done | 0.51s | 130 tokens | 254.9 t/s
done | 1.13s | 488 tokens | 431.9 t/s
done | 0.82s | 572 tokens | 697.6 t/s

Rank  Model                       Time   Tokens      Speed
-----------------------------------------------------------------
#1    LLaMA 3.1 8B              0.51s      130  254.9 t/s
#2    GPT-OSS 20B               0.82s      572  697.6 t/s
#3    Qwen3 32B                 1.13s      488  431.9 t/s
#4    LLaMA 3.3 70B             1.58s      190  120.3 t/s

ANSWERS:

[ LLaMA 3.1 8B ]
Based on the provided context, the main topic of this document appears to be related to genetics and genomics, specifically focusing on the Human Genome Project and its applications in medical genetics.

[Source 1: Genome annotation.pdf, Page 3] supports this conclusion, as it mentions the Human Genome Pro

In [50]:
# CELL 12: Aggregate Analytics
print("\n" + "=" * 65)
print("AGGREGATE PERFORMANCE — ALL QUERIES")
print("=" * 65)

from collections import defaultdict

agg = defaultdict(lambda: {"times": [], "tokens": [], "wins": 0})

for run in all_runs:
    for mid, res in run["results"].items():
        if not res["error"]:
            agg[mid]["times"].append(res["time"])
            agg[mid]["tokens"].append(res["tokens"])
    if run["ranked"]:
        agg[run["ranked"][0]]["wins"] += 1

total = len(all_runs)

# Summary table
print(f"\n{'Model':<24} {'Avg Time':>9} {'Avg Tokens':>12} {'Wins':>6} {'Win Rate':>10}")
print("-" * 68)
for mid in sorted(agg.keys(), key=lambda k: sum(agg[k]["times"]) / max(len(agg[k]["times"]), 1)):
    d     = agg[mid]
    avg_t = sum(d["times"])  / len(d["times"])  if d["times"]  else 0
    avg_k = sum(d["tokens"]) / len(d["tokens"]) if d["tokens"] else 0
    wr    = f"{d['wins']}/{total} ({int(d['wins']/total*100 if total else 0)}%)"
    print(f"{MODELS[mid]['label']:<24} {avg_t:>7.2f}s {avg_k:>12.0f} {d['wins']:>6} {wr:>10}")

# Key observations
fastest_mid   = min(agg.keys(), key=lambda k: sum(agg[k]["times"]) / max(len(agg[k]["times"]), 1))
most_verbose  = max(agg.keys(), key=lambda k: sum(agg[k]["tokens"]) / max(len(agg[k]["tokens"]), 1))
most_wins_mid = max(agg.keys(), key=lambda k: agg[k]["wins"])

print(f"\nKey Observations:")
print(f"  Fastest model  : {MODELS[fastest_mid]['label']}")
print(f"  Most verbose   : {MODELS[most_verbose]['label']}")
print(f"  Most wins      : {MODELS[most_wins_mid]['label']} ({agg[most_wins_mid]['wins']}/{total})")
print(f"  Total queries  : {total}")



AGGREGATE PERFORMANCE — ALL QUERIES

Model                     Avg Time   Avg Tokens   Wins   Win Rate
--------------------------------------------------------------------
LLaMA 3.1 8B                0.69s          210      3  3/4 (75%)
GPT-OSS 20B                 0.82s          563      0   0/4 (0%)
LLaMA 3.3 70B               1.04s          171      1  1/4 (25%)
Qwen3 32B                   1.41s          613      0   0/4 (0%)

Key Observations:
  Fastest model  : LLaMA 3.1 8B
  Most verbose   : Qwen3 32B
  Most wins      : LLaMA 3.1 8B (3/4)
  Total queries  : 4


## Summary & Conclusions

### What We Built
A complete RAG pipeline that:
1. **Extracts** text from PDFs using PyMuPDF (`fitz`)
2. **Chunks** text with configurable size and overlap
3. **Embeds** chunks using `all-MiniLM-L6-v2` (384-dim dense vectors)
4. **Indexes** with FAISS `IndexFlatL2` for exact nearest-neighbour search
5. **Retrieves** relevant context per query (diverse or top-K modes)
6. **Queries** 4 different LLMs with **identical context** → controlled comparison
7. **Analyzes** answers by speed, token count, and response verbosity

### Model Characteristics (Expected)
| Model | Typical Strength | Speed | Verbosity |
|-------|-----------------|-------|-----------|
| LLaMA 3.3 70B | Deep reasoning, long answers | Moderate | High |
| LLaMA 3.1 8B | Fast, good for simple lookups | Very fast | Medium |
| Qwen3 32B | Balanced quality & speed | Fast | Medium |
| GPT-OSS 20B | Concise factual answers | Fast | Low-Medium |

### Key Takeaway
> No single model wins on every dimension. The right LLM depends on whether your use-case prioritises **answer depth**, **response speed**, or **context window** for your specific document corpus.

### Streamlit App
The companion `app.py` provides a full interactive UI with:
- Real-time side-by-side comparison across all 4 models
- Visual metrics dashboard (response time bars, token counts)
- Cumulative analytics tab tracking win rates across queries
- Query history log
